<a href="https://colab.research.google.com/github/smin12345/NLP_Practice/blob/main/NLP_05.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5주차 실습: 텍스트 표현 방법

**주제:** Bag-of-Words(BoW), TF-IDF, n-gram 특징 추출

## 학습 목표
1. 토큰화된 텍스트를 수치 벡터로 표현해야 하는 이유를 설명한다.
2. CountVectorizer로 BoW 행렬을 만들고 희소행렬의 의미를 해석한다.
3. n-gram의 장점과 차원 증가·희소성의 한계를 판단한다.
4. TF-IDF의 TF·DF·IDF를 구분하고 과업에 맞게 설정한다.

> **제출:** 모든 TODO를 작성·실행하고 체크포인트 답변을 Markdown 셀에 남기세요. 외부 AI·자료를 사용했다면 사용 범위와 검증 과정을 기록하세요.

## 0. 준비
교육용 예시 데이터입니다. 실제 데이터는 출처·라이선스·개인정보(PII) 포함 여부를 확인하세요.

In [2]:
# 오류가 날 때만 아래 주석을 해제하세요.
!pip -q install -U scikit-learn pandas kiwipiepy

import pandas as pd
from kiwipiepy import Kiwi
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

kiwi = Kiwi()
corpus = [
    '배송이 빠르고 제품 품질이 좋아요',
    '배송은 느렸지만 제품 디자인은 예뻐요',
    '품질은 좋지만 가격이 조금 비싸요',
    '가격 대비 품질이 만족스러워요',
    '배송이 너무 느리고 포장도 아쉬워요'
]
df = pd.DataFrame({'doc_id': range(1, len(corpus)+1), 'text': corpus})
display(df)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 MB 8.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 54.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 36.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 34.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.


,doc_id,text
0,1,배송이 빠르고 제품 품질이 좋아요
1,2,배송은 느렸지만 제품 디자인은 예뻐요
2,3,품질은 좋지만 가격이 조금 비싸요
3,4,가격 대비 품질이 만족스러워요
4,5,배송이 너무 느리고 포장도 아쉬워요


## 1. 텍스트를 벡터로 표현하는 이유
머신러닝 모델은 문장 자체가 아니라 숫자 입력을 계산합니다. 4주차의 형태소·토큰을 열(feature)로 만들고, 문서마다 각 토큰의 값(빈도 또는 중요도)을 채웁니다.

`원문 → 정제·토큰화 → 문서-토큰 행렬 → 분류/예측`

In [3]:
# TODO 1: 공백 기준 토큰화로 CountVectorizer를 만들고 문서-토큰 행렬을 확인하세요.
# count_vec = ...
count_vec = CountVectorizer()
# X_count = ...
X_count = count_vec.fit_transform(corpus)
print(count_vec.get_feature_names_out())
display(pd.DataFrame(X_count.toarray(), columns=count_vec.get_feature_names_out()))

['가격' '가격이' '너무' '느렸지만' '느리고' '대비' '디자인은' '만족스러워요' '배송은' '배송이' '비싸요' '빠르고'
 '아쉬워요' '예뻐요' '제품' '조금' '좋아요' '좋지만' '포장도' '품질은' '품질이']


,가격,가격이,너무,느렸지만,느리고,대비,디자인은,만족스러워요,배송은,배송이,...,빠르고,아쉬워요,예뻐요,제품,조금,좋아요,좋지만,포장도,품질은,품질이
0,0,0,0,0,0,0,0,0,0,1,...,1,0,0,1,0,1,0,0,0,1
1,0,0,0,1,0,0,1,0,1,0,...,0,0,1,1,0,0,0,0,0,0
2,0,1,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,1,0,1,0
3,1,0,0,0,0,1,0,1,0,0,...,0,0,0,0,0,0,0,0,0,1
4,0,0,1,0,1,0,0,0,0,1,...,0,1,0,0,0,0,0,1,0,0


### 체크포인트 1
문서-토큰 행렬에서 **행**, **열**, **값**이 각각 무엇을 뜻하는지 1문장씩 설명하세요. 행렬의 0이 많은 이유도 설명하세요.

행은 각각의 문서를 의미하며, 이 결과에서는 리뷰 5개에 해당한다.
열은 전체 문서를 공백 기준으로 토큰화하여 얻은 서로 다른 토큰 21개를 의미한다.
값은 해당 문서에서 각 토큰이 등장한 횟수를 의미하며, 0은 등장하지 않았다는 뜻이다.
0이 많은 이유는 각 문서에 전체 토큰 중 일부만 등장하고, 공백 기준 토큰화에서는 ‘가격’과 ‘가격이’처럼 조사가 붙은 표현도 서로 다른 토큰으로 처리되기 때문이다.

## 2. Bag-of-Words(BoW)
BoW는 각 단어의 등장 횟수를 사용합니다. 구현은 간단하지만 단어 순서와 긴 문맥 정보를 잃습니다. `배송이 빠르다`와 `빠르지 않은 배송`처럼 순서·부정이 중요한 경우를 생각해 보세요.

In [6]:
#TODO 2: max_features=5를 적용한 BoW를 만들고, 어떤 단어가 남았는지 확인하세요.
small_count_vec = CountVectorizer(max_features=5)
X_small = small_count_vec.fit_transform(corpus)
print(small_count_vec.get_feature_names_out())

['가격' '느리고' '배송이' '제품' '품질이']


## 3. n-gram: 인접한 토큰 묶기
- unigram: `배송`, `빠르다`
- bigram: `배송 빠르다`
- trigram: `배송 이 빠르다`

n-gram은 일부 어순·표현 정보를 보완하지만, n이 커질수록 특징 수가 늘고 희소성이 커질 수 있습니다.

In [4]:
# TODO 3: unigram과 bigram을 함께 쓰는 CountVectorizer를 만드세요.
ngram_vec = CountVectorizer(ngram_range=(1, 2))
X_ngram = ngram_vec.fit_transform(corpus)
print(ngram_vec.get_feature_names_out())
print('특징 수:', X_ngram.shape[1])

['가격' '가격 대비' '가격이' '가격이 조금' '너무' '너무 느리고' '느렸지만' '느렸지만 제품' '느리고'
 '느리고 포장도' '대비' '대비 품질이' '디자인은' '디자인은 예뻐요' '만족스러워요' '배송은' '배송은 느렸지만' '배송이'
 '배송이 너무' '배송이 빠르고' '비싸요' '빠르고' '빠르고 제품' '아쉬워요' '예뻐요' '제품' '제품 디자인은'
 '제품 품질이' '조금' '조금 비싸요' '좋아요' '좋지만' '좋지만 가격이' '포장도' '포장도 아쉬워요' '품질은'
 '품질은 좋지만' '품질이' '품질이 만족스러워요' '품질이 좋아요']
특징 수: 40


### 체크포인트 2
감성 분석에서 bigram이 unigram보다 유용할 수 있는 표현을 하나 만들고, 그 이유를 설명하세요. 특징 수 증가가 항상 좋은 것은 아닌 이유도 함께 적으세요.

감성 분석에서 '안 좋아요'는 bigram이 유용한 표현이다. Unigram은 '안'과 '좋아요'를 각각 분리하지만, bigram은 '안 좋아요'를 하나의 특징으로 묶어 부정적인 의미를 파악하는 데 도움이 된다.
특징 수가 늘어나면 드물게 등장하는 표현이나 불필요한 특징도 많아져 데이터가 희소해지고, 계산량과 메모리 사용량이 증가할 수 있다.

## 4. TF-IDF: 많이 나오지만 덜 구별되는 단어의 가중치 낮추기
- **TF (Term Frequency):** 한 문서 안에서의 단어 빈도
- **DF (Document Frequency):** 해당 단어를 포함한 문서 수
- **IDF (Inverse Document Frequency):** 여러 문서에 흔한 단어의 가중치를 낮추는 값

TF-IDF는 단어의 중요도를 표현하는 한 방법이며, 과업에 따라 BoW가 더 적절할 수도 있습니다.

In [7]:
# TODO 4: TfidfVectorizer로 TF-IDF 행렬을 만들고 소수점 둘째 자리까지 확인하세요.
tfidf_vec = TfidfVectorizer()
X_tfidf = tfidf_vec.fit_transform(corpus)
tfidf_df = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_vec.get_feature_names_out())
display(tfidf_df.round(2))

,가격,가격이,너무,느렸지만,느리고,대비,디자인은,만족스러워요,배송은,배송이,...,빠르고,아쉬워요,예뻐요,제품,조금,좋아요,좋지만,포장도,품질은,품질이
0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.41,...,0.5,0.00,0.00,0.41,0.00,0.5,0.00,0.00,0.00,0.41
1,0.00,0.00,0.00,0.46,0.00,0.00,0.46,0.00,0.46,0.00,...,0.0,0.00,0.46,0.37,0.00,0.0,0.00,0.00,0.00,0.00
2,0.00,0.45,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,...,0.0,0.00,0.00,0.00,0.45,0.0,0.45,0.00,0.45,0.00
3,0.52,0.00,0.00,0.00,0.00,0.52,0.00,0.52,0.00,0.00,...,0.0,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.42
4,0.00,0.00,0.46,0.00,0.46,0.00,0.00,0.00,0.00,0.37,...,0.0,0.46,0.00,0.00,0.00,0.0,0.00,0.46,0.00,0.00


In [8]:
# TODO 5: 아래 문서에서 BoW와 TF-IDF의 상위 3개 토큰을 비교하세요.
target_idx = 1
count_scores = pd.Series(X_count.toarray()[target_idx], index=count_vec.get_feature_names_out())
tfidf_scores = pd.Series(X_tfidf.toarray()[target_idx], index=tfidf_vec.get_feature_names_out())
print('BoW 상위:', count_scores.sort_values(ascending=False).head(3))
print('TF-IDF 상위:', tfidf_scores.sort_values(ascending=False).head(3))

BoW 상위: 느렸지만    1
예뻐요     1
제품      1
dtype: int64
TF-IDF 상위: 느렸지만    0.463693
배송은     0.463693
예뻐요     0.463693
dtype: float64


### 체크포인트 3
여러 문서에 공통으로 등장하는 단어는 TF-IDF에서 일반적으로 어떤 영향을 받나요? 이 단어가 과업상 중요할 수도 있는 사례를 하나 드세요.

여러 문서에 공통으로 등장하는 단어는 IDF 값이 낮아져, 등장 횟수가 같다면 드물게 등장하는 단어보다 TF-IDF 가중치가 낮아진다.
예를 들어, '배송'은 3개 문서에 등장하여 가중치가 낮아질 수 있다. 하지만 상품 리뷰에서 배송 속도에 대한 만족이나 불만을 파악하는 과업에서는 '배송'이 중요한 키워드가 될 수 있다.

## 5. 한국어 형태소 토큰을 벡터화하기
공백 토큰화 대신 4주차의 형태소 분석 결과를 벡터화할 수 있습니다. 여기서는 키워드 추출 예시로 명사 계열을 사용합니다. 감성 분석이라면 용언·부사·부정 표현 보존을 추가로 검토해야 합니다.

In [9]:
def kiwi_noun_tokens(text):
    return [token.form for token in kiwi.tokenize(text)
            if token.tag.startswith('N') and len(token.form) > 1]

print('예시 토큰:', kiwi_noun_tokens(corpus[0]))

# TODO 6: tokenizer에 kiwi_noun_tokens를 연결한 TfidfVectorizer를 만들고 결과를 확인하세요.
korean_tfidf = TfidfVectorizer(tokenizer=kiwi_noun_tokens, token_pattern=None)
X_korean = korean_tfidf.fit_transform(corpus)
print(korean_tfidf.get_feature_names_out())
display(pd.DataFrame(X_korean.toarray(), columns=korean_tfidf.get_feature_names_out()).round(2))

예시 토큰: ['배송', '제품', '품질']
['가격' '대비' '디자인' '만족' '배송' '제품' '포장' '품질']


,가격,대비,디자인,만족,배송,제품,포장,품질
0,0.00,0.00,0.00,0.00,0.54,0.65,0.00,0.54
1,0.00,0.00,0.69,0.00,0.46,0.56,0.00,0.00
2,0.77,0.00,0.00,0.00,0.00,0.00,0.00,0.64
3,0.46,0.57,0.00,0.57,0.00,0.00,0.00,0.38
4,0.00,0.00,0.00,0.00,0.56,0.00,0.83,0.00


## 6. 미니 과제: 특징 추출 정책 설계
다음 중 하나를 선택하세요. (1) 강의평 키워드 추출 (2) 상품 리뷰 감성 분석 (3) 학교 민원 자동 분류

분석기/토큰 단위, 품사 선택, n-gram 범위, 최소 문서 빈도(`min_df`), 사용할 표현(BoW 또는 TF-IDF)을 근거와 함께 결정하세요.

In [15]:
# TODO 7: 선택한 과업에 맞게 아래 설정을 수정하고 실행하세요.

corpus = [
    "교수님이 어려운 개념을 예시와 함께 설명해 주셔서 이해하기 쉬웠다",
    "과제가 너무 많고 제출 기한이 짧아서 부담스러웠다",
    "실습을 통해 직접 코드를 작성할 수 있어서 도움이 되었다",
    "수업 진행 속도가 빠르고 시험 범위가 넓어서 따라가기 어려웠다",
    "시험 범위를 명확하게 알려 주시고 질문에도 친절하게 답해 주셨다"
]

df = pd.DataFrame({
    'doc_id': range(1, len(corpus) + 1),
    'text': corpus
})
display(df)

my_vectorizer = TfidfVectorizer(
    tokenizer=kiwi_noun_tokens,
    token_pattern=None,
    ngram_range=(1, 1),
    min_df=1
)

X_my = my_vectorizer.fit_transform(corpus)

print('특징 수:', X_my.shape[1])
print('특징:', my_vectorizer.get_feature_names_out())

my_tfidf_df = pd.DataFrame(
    X_my.toarray(),
    index=df['doc_id'],
    columns=my_vectorizer.get_feature_names_out()
)
display(my_tfidf_df.round(2))


,doc_id,text
0,1,교수님이 어려운 개념을 예시와 함께 설명해 주셔서 이해하기 쉬웠다
1,2,과제가 너무 많고 제출 기한이 짧아서 부담스러웠다
2,3,실습을 통해 직접 코드를 작성할 수 있어서 도움이 되었다
3,4,수업 진행 속도가 빠르고 시험 범위가 넓어서 따라가기 어려웠다
4,5,시험 범위를 명확하게 알려 주시고 질문에도 친절하게 답해 주셨다


특징 수: 20
특징: ['개념' '과제' '교수' '기한' '도움' '범위' '부담' '설명' '속도' '수업' '시험' '실습' '예시' '이해'
 '작성' '제출' '진행' '질문' '친절' '코드']


,개념,과제,교수,기한,도움,범위,부담,설명,속도,수업,시험,실습,예시,이해,작성,제출,진행,질문,친절,코드
doc_id,,,,,,,,,,,,,,,,,,,,
1,0.45,0.0,0.45,0.0,0.0,0.00,0.0,0.45,0.00,0.00,0.00,0.0,0.45,0.45,0.0,0.0,0.00,0.00,0.00,0.0
2,0.00,0.5,0.00,0.5,0.0,0.00,0.5,0.00,0.00,0.00,0.00,0.0,0.00,0.00,0.0,0.5,0.00,0.00,0.00,0.0
3,0.00,0.0,0.00,0.0,0.5,0.00,0.0,0.00,0.00,0.00,0.00,0.5,0.00,0.00,0.5,0.0,0.00,0.00,0.00,0.5
4,0.00,0.0,0.00,0.0,0.0,0.39,0.0,0.00,0.48,0.48,0.39,0.0,0.00,0.00,0.0,0.0,0.48,0.00,0.00,0.0
5,0.00,0.0,0.00,0.0,0.0,0.44,0.0,0.00,0.00,0.00,0.44,0.0,0.00,0.00,0.0,0.0,0.00,0.55,0.55,0.0


### 체크포인트 4
선택한 과업의 특징 추출 정책을 설명하세요. 다음을 모두 포함합니다.
1. BoW/TF-IDF 선택 이유  2. 토큰·품사 선택  3. n-gram과 min_df 설정  4. 예상 오류 또는 정보 손실 사례

## 제출 체크리스트
- [ ] TODO 1~7을 모두 작성하고 실행했다.
- [ ] BoW 행렬의 행·열·값·희소성을 설명했다.
- [ ] unigram과 bigram의 차이·한계를 설명했다.
- [ ] TF·DF·IDF의 역할을 설명했다.
- [ ] 공백 기반과 형태소 기반 벡터화를 비교했다.
- [ ] 선택 과업에 맞는 특징 추출 정책과 오류 사례를 기록했다.
- [ ] 외부 AI/자료 사용 시 사용 범위와 검증 과정을 기록했다.

## 다음 주차 예고
다음 시간에는 만들어진 문서-토큰 행렬을 입력으로 사용하여 나이브 베이즈와 로지스틱 회귀 기반 텍스트 분류 모델을 학습하고 평가합니다.

BoW/TF-IDF 선택 이유: 강의평 키워드 추출을 위해 TF-IDF를 선택하였다. 단순 등장 횟수를 사용하는 BoW보다 여러 문서에 공통으로 등장하는 단어의 비중을 낮추고, 각 강의평의 특징적인 키워드를 파악하는 데 적합하기 때문이다.

토큰·품사 선택: Kiwi로 형태소 분석을 수행하고 두 글자 이상의 명사류만 추출하였다. 조사를 제거하여 같은 단어를 하나의 특징으로 처리하고, ‘과제’, ‘실습’, ‘시험’ 등 주요 주제를 나타내는 단어를 남기기 위해서이다.

n-gram과 min_df 설정: 개별 키워드를 추출하기 위해 ngram_range=(1, 1)로 설정하였다. 문서가 5개로 적으므로 한 문서에만 등장하는 키워드도 유지하도록 min_df=1로 설정하였다.

예상 오류 또는 정보 손실 사례: 명사만 추출하면 ‘과제가 많다’와 ‘과제가 적다’가 모두 ‘과제’로 표현되어 평가의 차이가 사라질 수 있다.

In [11]:
Question01.IDF가 높은 단어일수록 전체 코퍼스내에서 자주 등장하는 흔한 단어이다? (X / O)
X — IDF가 높은 단어는 전체 코퍼스에서 등장하는 문서 수가 적은 단어이다.
Question02.BoW의 가장 큰 한계점인 '단어순서정보유실'을 보완하기 위해 사용하는 기법은?
n-gram — 연속된 단어를 묶어 특징으로 사용하여 단어의 순서 정보를 일부 반영한다. 예를 들어 bigram은 ‘안 좋아요’를 하나의 특징으로 처리한다.